# Un Transformer *from scratch* sur son propre corpus

Ce notebook entraîne de bout en bout un **GPT** (un Transformer décodeur) sur un corpus de texte, avec le code de la séance 2 du dépôt [Creative Technology](https://github.com/nathanvid/Creative-Technology) :

| Étape | Code repris de |
|---|---|
| Lecture du corpus, vocabulaire caractère, `encode` / `decode`, découpage train / validation | `01_texte_vers_nombres` |
| Tokenizer BPE : octets UTF-8, fusions, pré-découpage façon GPT-2 (avec la bibliothèque `tokenizers` de Hugging Face) | `08_bpe` |
| `Dataset` / `DataLoader`, entraînement compté en pas, validation tous les `eval_every` pas | `06_transformer` |
| Tête d'attention masquée, multi-têtes, feed-forward, blocs pre-norm avec résiduels, GPT | `05_self_attention`, `06_transformer`, `07_generation` |
| Génération : glouton, température, top-k, top-p, mesures de qualité | `07_generation` |
| Checkpoint : poids + configuration + vocabulaire | `06_transformer`, `07_generation` |
| Architecture moderne (RoPE, RMSNorm, QK-norm, ReLU², connexions en U), optimiseur Muon / NorMuon, planning WSD | le speedrun [modded-nanoGPT](https://github.com/KellerJordan/modded-nanogpt), la recette de notre modèle final `v6-591M` |

**Le modèle est celui que nous avons finalement entraîné**, `v6-591M` (591 M paramètres, pré-entraîné sur TPU puis affiné sur les Simpson) : le même code, `ModernGPT`, avec la même recette (NorMuon, planning WSD, pas de dropout). Seule la taille change : le préréglage `v6` reprend ses dimensions exactes, les autres le réduisent pour qu'il s'entraîne sur un Mac. Le GPT du cours (`GPTLanguageModel`) reste dans `gpt.py`, comme référence (section 5).

Le modèle, les tokenizers et la génération sont écrits une seule fois, dans `src/super_modele/gpt.py` : ce notebook les importe (et en affiche le code), comme le chat (`chat.py`) et l'interface (`app.py`).

Par-dessus, ce qu'il faut pour un vrai entraînement : un pré-entraînement sur un gros corpus général (Wikipédia en français) avant l'affinage sur ses dialogues, des préréglages de taille, un learning rate en trois temps (montée, plateau, descente : le planning WSD), la précision mixte bf16, `torch.compile`, l'arrêt anticipé, la sauvegarde du meilleur modèle, l'arrêt propre en cours de route, et les bits par caractère pour comparer les runs. L'outillage vient de bibliothèques : 🤗 **datasets** (lecture et encodage des corpus), **ftfy** (nettoyage du texte), **TensorBoard** (suivi de l'entraînement), **safetensors** (checkpoints), **Gradio** (interface).

**Mode d'emploi**

1. Déposer le corpus dans `data/` : un `.txt`, un `.csv`, un `.jsonl` / `.json`, un `.parquet`, un dossier de fichiers, ou donner une URL.
2. Dans la cellule **Configuration**, mettre son chemin dans `DATA_PATH` (`None` : essai sur Tiny Shakespeare, téléchargé automatiquement). Le notebook est réglé pour les scripts des Simpson en français, convertis par `prepare_simpsons.py` en dialogues `Personnage: réplique` (473 épisodes, 103 000 répliques).
3. *Run All*. Pour suivre l'entraînement en direct : `uv run tensorboard --logdir runs`, puis http://localhost:6006. Le meilleur modèle est sauvegardé dans `checkpoints/`.
4. Pour discuter avec le modèle : `uv run app.py`, puis http://localhost:8765.
5. La section 9 recharge aussi `v6-591M-ft2`, le modèle entraîné sur TPU, s'il est dans `checkpoints/`.

L'entraînement peut être interrompu à tout moment (bouton ■ du notebook) : le meilleur modèle déjà mesuré est gardé, et les cellules suivantes fonctionnent avec lui.

In [ ]:
import inspect
import math
import os
import random
import re
import shutil
import sys
import time
from collections import Counter
from pathlib import Path

# tokenizers (Hugging Face) encode en parallèle ; le dire explicitement évite un avertissement quand torch.compile
# lance ses processus de compilation (qui n'utilisent pas le tokenizer).
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

import ftfy  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402
from datasets import load_dataset  # noqa: E402
from IPython.display import Code, display  # noqa: E402
from torch import nn  # noqa: E402
from torch.utils.data import DataLoader, Dataset, RandomSampler  # noqa: E402
from torch.utils.tensorboard import SummaryWriter  # noqa: E402

## 0. Configuration

Tous les réglages sont dans la cellule suivante. Ceux du modèle et de l'entraînement viennent d'un **préréglage**. Tous ont l'architecture de `v6-591M`, seule la taille change. Les nombres de paramètres sont donnés pour un vocabulaire BPE de 8 192 tokens :

| Préréglage | Paramètres | Contexte | Pas | Durée (Mac M5, sans pré-entraînement) | Pour un corpus de |
|---|---|---|---|---|---|
| `test` | 2,9 M | 64 | 1 500 | ~1 min | vérifier que tout marche |
| `petit` | 4,9 M | 128 | 4 000 | ~7 min | moins de 250 000 tokens |
| `moyen` | 8,9 M | 128 | 5 000 | ~15 min | 250 000 à 2,5 M tokens |
| `gros` | 16,9 M, les dimensions du `gpt.py` de Karpathy | 256 | 5 000 | ~50 min | plus de 2,5 M tokens |
| `v6` | 591 M, **notre modèle final** : 20 blocs de 1 536, 12 têtes de 128 | 512 | — | entraîné sur TPU : 7,0 G tokens à ~95 000 tokens/s | des milliards de tokens |

Avec `gros`, le pré-entraînement sur Wikipédia ajoute environ 1 h 25 (4 500 pas de 1,1 s). Le préréglage `v6` sert à voir l'architecture exacte du modèle final : sur un Mac, un pas prend plusieurs secondes. Il s'est entraîné avec `kaggle/train_tpu.py`, sur des lots de 524 288 tokens.

`PRESET = "auto"` choisit d'après le nombre de tokens d'entraînement (en mode caractère, 1 token = 1 caractère). Pour changer un réglage sans toucher au reste : `OVERRIDES = {"max_steps": 8_000, "lr_scale": 0.5}`.

In [ ]:
# --- Données -------------------------------------------------------------------
DATA_PATH = "data/simpsons_scripts/dialogues.jsonl"  # fichier (.txt .csv .jsonl .json .parquet), dossier ou URL ; None = Shakespeare
TEXT_COLUMN = "text"  # .csv / .jsonl / .json / .parquet : la colonne qui contient le texte
ENCODING = "utf-8"  # encodage des .txt et des .csv, ex. "cp1252" pour un vieil export Windows
# Mise en forme de chaque document : des substitutions (expression régulière, remplacement), dans l'ordre ;
# [] = texte tel quel. Ex. pour le dataset HomerSimpson de Hugging Face, « ### Human: …### Assistant: … » :
# [(r"^### Human:\s*", "Human: "), (r"\s*### Assistant:\s*", "\nHomer: "), (r"\s*### Human:\s*$", "")]
DOC_REPLACEMENTS = []
DOC_SEPARATOR = "\n\n"  # inséré entre deux documents (fichiers, lignes d'un CSV…) ; ici entre deux épisodes
SHUFFLE_DOCUMENTS = True  # mélange les documents avant le découpage train / val (s'il y en a plusieurs)
MIN_CHAR_COUNT = 1  # supprime les caractères vus moins de N fois dans le corpus (1 = tout garder)
VAL_FRACTION = 0.1  # validation : mesurée pendant l'entraînement, pour garder le meilleur modèle (script 01)
TEST_FRACTION = 0.1  # test : la toute fin du corpus, mesurée une seule fois à la fin (0 = pas de test)

# --- Pré-entraînement ----------------------------------------------------------
# Le modèle apprend d'abord la langue sur un gros corpus général, puis s'affine sur DATA_PATH (section 6).
# Ici trois des 17 fichiers de Wikipédia en français (licences CC BY-SA 3.0 et GFDL, ~700 Mo), lus par
# 🤗 datasets : téléchargés une fois, puis gardés en cache.
PRETRAIN_NAME = "wikipedia_fr"  # nom du corpus général (checkpoint, TensorBoard)
PRETRAIN_FILES = [f"hf://datasets/wikimedia/wikipedia/20231101.fr/train-{i:05d}-of-00017.parquet" for i in (4, 12, 13)]
PRETRAIN_DATA = dict(path="parquet", data_files=PRETRAIN_FILES)  # arguments de load_dataset ; None = pas de pré-entraînement
PRETRAIN_MAX_CHARS = 650_000_000  # on garde 650 millions de caractères, ~190 M tokens
# Lots de 128 (le GPU travaille mieux), et pas de dropout : chaque texte n'est vu qu'une fois, le modèle ne
# peut pas le réciter. 4 500 pas × 128 × 256 tokens ≈ 147 M tokens, ~1 h 25 sur Mac M5.
PRETRAIN = dict(max_steps=4_500, batch_size=128, dropout=0.0, val_batches=20)
PRETRAIN_VAL_FRACTION = 0.005  # la petite part du corpus général qui sert à suivre le pré-entraînement
# L'affinage, comme pour v6-591M : les learning rates × 0,1, une courte montée (5 %) puis une descente jusqu'à 0.
FINETUNE = dict(lr_scale=0.1, warmup=0.05, cooldown=0.95, max_steps=3_000)

# --- Génération ----------------------------------------------------------------
# Pour discuter avec le modèle (section 8) : {me} parle, {character} répond. None si le corpus n'est pas un dialogue.
CHAT_TEMPLATE = "{me}: {message}\n{character}:"
CHAT_ME = "Homme"  # qui parle : un personnage du corpus (« Homme » = un passant anonyme), ex. "Marge", "Bart"
CHAT_CHARACTER = "Homer"  # à qui on parle, ex. "Marge", "Bart", "Lisa", "Burns", "Moe", "Ned"

# --- Tokenizer -----------------------------------------------------------------
TOKENIZER = "bpe"  # "char" : un token = un caractère (script 01) ; "bpe" : sous-mots (script 08, Hugging Face)
BPE_VOCAB_SIZE = 8192  # taille du vocabulaire BPE (≥ 256), ignorée en mode "char" ; plus de données → plus grand

# --- Modèle et entraînement ----------------------------------------------------
PRESET = "gros"  # "test", "petit", "moyen", "gros", ou "auto" (choisi d'après la taille du corpus)
OVERRIDES = {}  # ex. {"max_steps": 8_000, "lr_scale": 0.5}
USE_AMP = True  # précision mixte bf16 sur GPU / Mac : plus rapide
COMPILE = True  # torch.compile : ~2× plus rapide sur Mac ; s'il échoue, l'entraînement continue sans
DEVICE = None  # None = le GPU s'il y en a un ; "cpu" pour forcer le processeur
RUN_NAME = None  # nom du checkpoint et du run TensorBoard ; None = construit automatiquement

PRESETS = {  # tous en ModernGPT, l'architecture de v6-591M, sans dropout comme lui
    "test": dict(block_size=64, n_embd=128, n_head=4, n_layer=4, dropout=0.0, batch_size=32, max_steps=1_500),
    "petit": dict(block_size=128, n_embd=192, n_head=4, n_layer=4, dropout=0.0, batch_size=64, max_steps=4_000),
    "moyen": dict(block_size=128, n_embd=256, n_head=4, n_layer=6, dropout=0.0, batch_size=64, max_steps=5_000),
    "gros": dict(block_size=256, n_embd=384, n_head=6, n_layer=6, dropout=0.0, batch_size=64, max_steps=5_000),
    # Le modèle final, à l'identique. Il s'est entraîné sur TPU (kaggle/train_tpu.py) : 13 358 pas de 1 024 × 512
    # tokens. Sur un Mac, un pas prend plusieurs secondes : pour voir l'architecture, pas pour l'entraîner ici.
    "v6": dict(block_size=512, n_embd=1536, n_head=12, n_layer=20, dropout=0.0, batch_size=8, max_steps=5_000),
}
# L'optimisation de v6-591M, commune à tous les préréglages : les learning rates du speedrun modded-nanoGPT.
TRAINING_DEFAULTS = dict(
    muon_lr=0.05,  # NorMuon, pour les matrices des blocs
    embed_lr=0.6,  # Adam, pour l'embedding des tokens
    head_lr=0.008,  # Adam, pour la tête (les logits)
    scalar_lr=0.04,  # Adam, pour les poids des connexions en U
    normuon=True,  # False : Muon simple, sans l'échelle par neurone de NorMuon
    lr_scale=1.0,  # multiplie les quatre learning rates (0,1 pour l'affinage)
    warmup=0.01,  # planning WSD : montée sur le premier 1 % des pas…
    cooldown=0.4,  # …plateau, puis descente linéaire jusqu'à 0 sur les derniers 40 %
    eval_every=250,  # une mesure de validation tous les 250 pas (script 06)
    val_batches=50,  # lots de validation par mesure
    patience=6,  # arrêt anticipé après 6 mesures (1 500 pas) sans amélioration de la validation
)

In [ ]:
SEED = 1337
torch.manual_seed(SEED)

try:
    ROOT = Path(__file__).resolve().parent
except NameError:  # exécution dans une cellule Jupyter : `__file__` n'existe pas
    ROOT = Path.cwd()

# Le modèle, les tokenizers, la génération et les checkpoints sont écrits une seule fois, dans
# src/super_modele/gpt.py : ce notebook, le chat (chat.py) et l'interface (app.py) importent le même code.
# On ajoute src/ au chemin d'import à la main : dans un dossier synchronisé par iCloud, macOS cache les
# fichiers de .venv, et Python ignore alors le .pth de l'installation éditable du paquet.
sys.path.insert(0, str(ROOT / "src"))
from super_modele.gpt import (  # noqa: E402
    BPETokenizer, CharTokenizer, ModernAttention, ModernBlock, ModernGPT, ModernMLP, Muon, Rotary,
    generate, load_checkpoint, modern_optimizers, newton_schulz, reply, rms_norm, save_checkpoint, wsd_schedule,
)

CHECKPOINT_DIR = ROOT / "checkpoints"
RUNS_DIR = ROOT / "runs"  # les runs TensorBoard
SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

device = DEVICE or (torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu")
# bf16 : la même plage de valeurs que le float32, avec moins de chiffres significatifs (pas besoin de GradScaler).
amp = USE_AMP and device in ("cuda", "mps") and (device != "cuda" or torch.cuda.is_bf16_supported())
print(f"torch {torch.__version__}   device : {device}   précision mixte bf16 : {amp}")


def autocast():
    """Contexte de précision mixte : les produits matriciels en bf16, le reste en float32."""
    return torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=amp)


def fmt_duration(seconds: float) -> str:
    """Durée lisible : `42s`, `3m05s`, `1h07`."""
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}h{m:02d}" if h else f"{m}m{s:02d}s" if m else f"{s}s"

## 1. Le corpus

Un modèle de langue n'a besoin **que d'une longue suite de caractères**, sans étiquette : la cible, c'est le texte lui-même décalé d'un cran (script 01). On lit les documents avec 🤗 **datasets** (un document par fichier texte, par ligne d'un CSV ou d'un JSONL ; un fichier, un dossier ou une URL), et on les met bout à bout, séparés par `DOC_SEPARATOR`.

Le nettoyage reste minimal, pour que le modèle apprenne le texte tel qu'il est :

- **ftfy** répare les erreurs d'encodage (`Ã©` au lieu de `é`, guillemets Windows mal décodés…) et normalise l'Unicode en NFC : `é` peut s'écrire en un caractère ou en deux (`e` + accent combinant), on n'en garde qu'une écriture ;
- les espaces insécables (fréquentes en français devant `:` `;` `!` `?` `»`) deviennent des espaces simples ;
- `DOC_REPLACEMENTS` applique, si besoin, des substitutions propres au corpus.

In [ ]:
SPACE_TABLE = str.maketrans({" ": " ", " ": " ", " ": " "})


def clean_text(text_: str) -> str:
    """ftfy répare l'encodage et normalise en NFC ; les espaces insécables deviennent des espaces simples."""
    return ftfy.fix_text(text_).translate(SPACE_TABLE)


def format_document(doc: str) -> str:
    """Applique les substitutions `DOC_REPLACEMENTS`, dans l'ordre."""
    for pattern, replacement in DOC_REPLACEMENTS:
        doc = re.sub(pattern, replacement, doc)
    return doc


def load_texts(source: str, text_column: str) -> list[str]:
    """Les documents d'un fichier, d'un dossier ou d'une URL, lus par 🤗 datasets.

    Un document par fichier texte (`.txt`, `.md`), par ligne d'un CSV (séparateur détecté) ou d'un JSONL,
    par élément d'un JSON ou par ligne d'un Parquet ; `text_column` désigne la colonne du texte.
    """
    files = [source]
    if Path(source).is_dir():
        files = sorted(str(p) for p in Path(source).rglob("*") if p.is_file() and not p.name.startswith("."))
    suffix = Path(files[0].split("?")[0]).suffix.lower()
    files = [f for f in files if Path(f.split("?")[0]).suffix.lower() == suffix]
    if suffix in (".csv", ".tsv"):
        ds = load_dataset("csv", data_files=files, sep=None, engine="python", encoding=ENCODING, split="train")
    elif suffix in (".json", ".jsonl"):
        ds = load_dataset("json", data_files=files, split="train")
    elif suffix == ".parquet":
        ds = load_dataset("parquet", data_files=files, split="train")
    else:
        ds = load_dataset("text", data_files=files, sample_by="document", encoding=ENCODING, split="train")
        text_column = "text"
    if text_column not in ds.column_names:
        raise KeyError(f"pas de colonne {text_column!r} (colonnes : {ds.column_names}) : régler TEXT_COLUMN")
    return [t or "" for t in ds[text_column]]

In [ ]:
if DATA_PATH is None:
    corpus_name, corpus_source = "tinyshakespeare", SHAKESPEARE_URL
else:
    is_url = str(DATA_PATH).startswith(("http://", "https://", "hf://"))
    corpus_source = str(DATA_PATH) if is_url else str(ROOT / Path(DATA_PATH).expanduser())
    corpus_name = Path(str(DATA_PATH).split("?")[0]).stem
    if not is_url and not Path(corpus_source).exists():
        raise FileNotFoundError(f"{corpus_source} introuvable : vérifier DATA_PATH")

documents = [format_document(clean_text(doc)) for doc in load_texts(corpus_source, TEXT_COLUMN)]
documents = [doc for doc in documents if doc.strip()]  # on retire les documents vides
if SHUFFLE_DOCUMENTS and len(documents) > 1:
    random.Random(SEED).shuffle(documents)  # sinon la validation ne contiendrait que les derniers documents
text = DOC_SEPARATOR.join(documents)

if MIN_CHAR_COUNT > 1:
    counts = Counter(text)
    rare = sorted(c for c, n in counts.items() if n < MIN_CHAR_COUNT)
    text = text.translate({ord(c): None for c in rare})
    print(f"{len(rare)} caractères rares supprimés : {''.join(rare)!r}")

char_counts = Counter(text)
n_lines = text.count("\n") + 1
print(f"corpus : {corpus_source}")
print(f"{len(documents):,} documents   {len(text):,} caractères   {len(text.split()):,} mots   {n_lines:,} lignes")
print(f"{len(char_counts)} caractères distincts : {''.join(sorted(char_counts))!r}")
print("les plus rares : " + "   ".join(f"{c!r} ×{n}" for c, n in char_counts.most_common()[:-11:-1]))
print("-" * 80)
print(text[:500])

### Le corpus de pré-entraînement

Nos dialogues font quelques millions de tokens : de quoi apprendre un style, pas une langue. On les complète avec un **gros corpus général**, des articles de Wikipédia en français, sur lequel le modèle apprend d'abord le français avant de s'affiner sur les dialogues (section 6). 🤗 datasets télécharge les fichiers une fois et garde chaque étape en cache : on retire les sections de fin (références, liens externes…) et les ébauches de moins de 500 caractères, on mélange les articles et on en garde `PRETRAIN_MAX_CHARS` caractères. Le premier passage prend une ou deux minutes, les suivants quelques secondes.

In [ ]:
# Sections de fin d'article : des listes de références et de liens, pas de la langue.
END_SECTIONS = re.compile(r"^(Notes et références|Références|Notes|Voir aussi|Liens externes|Bibliographie|"
                          r"Articles connexes|Annexes|Sources|Filmographie|Discographie)\s*$", re.MULTILINE)


def clean_article(text_: str) -> str:
    """Un article sans ses sections de fin, sans lignes vides en trop et sans espaces insécables."""
    end = END_SECTIONS.search(text_)
    text_ = text_[: end.start()] if end else text_
    return re.sub(r"\n{3,}", "\n\n", text_).strip().translate(SPACE_TABLE)


pretrain = None
if PRETRAIN_DATA is not None:
    if TOKENIZER != "bpe":
        raise ValueError("le pré-entraînement demande TOKENIZER = 'bpe' : des caractères ne couvrent pas un corpus général")
    t0 = time.perf_counter()
    pretrain = load_dataset(**PRETRAIN_DATA, split="train")
    pretrain = pretrain.map(lambda batch: {"text": [clean_article(t) for t in batch["text"]]}, batched=True,
                            remove_columns=[c for c in pretrain.column_names if c != "text"], desc="nettoyage")
    pretrain = pretrain.filter(lambda batch: [len(t) >= 500 for t in batch["text"]], batched=True, desc="ébauches")
    pretrain = pretrain.shuffle(seed=SEED)
    lengths = np.cumsum(pretrain.map(lambda batch: {"n": [len(t) for t in batch["text"]]}, batched=True,
                                     remove_columns=["text"], desc="longueurs")["n"])
    n_docs = min(len(lengths), int(np.searchsorted(lengths, PRETRAIN_MAX_CHARS)) + 1)
    pretrain = pretrain.select(range(n_docs)).flatten_indices()
    pretrain_chars = int(lengths[n_docs - 1])
    print(f"pré-entraînement : {PRETRAIN_NAME}   {n_docs:,} documents   {pretrain_chars:,} caractères   "
          f"({time.perf_counter() - t0:.0f} s)")
    print("-" * 80)
    print(pretrain[0]["text"][:300])
else:
    print("pas de pré-entraînement (PRETRAIN_DATA = None)")

## 2. Train, validation et test

Comme au script 01, on découpe **sans mélanger** les caractères (ce qui détruirait le texte), en trois morceaux qui se suivent :

- **train** (80 %) : le seul texte sur lequel le modèle apprend ;
- **validation** (10 %) : mesurée pendant l'entraînement. Elle sert à **décider** : garder le meilleur checkpoint, arrêter quand elle ne progresse plus, comparer des réglages. À force de choisir d'après elle, son score devient un peu optimiste ;
- **test** (10 %) : mis de côté et mesuré **une seule fois**, à la fin, sur le modèle choisi. Comme aucune décision ne s'appuie sur lui, c'est le score honnête du modèle sur un texte jamais vu.

On découpe le texte **avant** de le tokeniser, pour que le tokenizer BPE n'apprenne ses fusions que sur le train.

In [ ]:
n_test = int(TEST_FRACTION * len(text))
n_train = len(text) - n_test - int(VAL_FRACTION * len(text))
train_text, val_text, test_text = text[:n_train], text[n_train : len(text) - n_test], text[len(text) - n_test :]
print(f"train : {len(train_text):,} caractères   val : {len(val_text):,}   test : {len(test_text):,}")

## 3. Le tokenizer

Deux choix, avec la même interface (`encode`, `decode`, `vocab_size`) :

- **`"char"`** : un token = un caractère (script 01). Vocabulaire minuscule, aucun mot inconnu, mais des séquences longues : avec un contexte de 128 tokens, le modèle voit 128 caractères, une vingtaine de mots.
- **`"bpe"`** : des sous-mots appris par *Byte Pair Encoding* (script 08). On part des 256 octets UTF-8 et on fusionne à répétition la paire la plus fréquente. Le même contexte couvre 2 à 3 fois plus de texte, au prix d'un vocabulaire plus grand (plus d'embeddings, et chaque token est plus rare dans les données). On l'entraîne avec la bibliothèque `tokenizers` de Hugging Face : le même algorithme que le script 08, écrit en Rust, donc des milliers de fois plus rapide.

Les deux classes sont dans `src/super_modele/gpt.py`. En mode caractère, le vocabulaire est construit sur **tout** le texte : un caractère qui n'apparaîtrait que dans la validation doit quand même avoir un indice. Le BPE n'apprend ses fusions que sur le train : partir des octets lui permet d'écrire n'importe quel texte.

In [ ]:
display(Code(inspect.getsource(CharTokenizer), language="python"))
display(Code(inspect.getsource(BPETokenizer), language="python"))

In [ ]:
if TOKENIZER == "char":
    tokenizer = CharTokenizer.train([text])
elif TOKENIZER == "bpe":
    t0 = time.perf_counter()
    # Le vocabulaire doit servir aux deux corpus : on l'apprend sur le train et sur ~100 M caractères du corpus général.
    sample, size = [train_text], 0
    if pretrain is not None:
        for batch in pretrain.iter(batch_size=1_000):
            sample.extend(batch["text"])
            size += sum(map(len, batch["text"]))
            if size >= 100_000_000:
                break
    tokenizer = BPETokenizer.train(sample, BPE_VOCAB_SIZE)
    print(f"{tokenizer.vocab_size - 256} fusions apprises en {time.perf_counter() - t0:.1f} s")
    # Les tokens 256, 257… sont les premières fusions apprises : les morceaux les plus fréquents du corpus.
    print("premières fusions : " + "  ".join(repr(tokenizer.decode([i])) for i in range(256, 276)))
else:
    raise ValueError(f"TOKENIZER inconnu : {TOKENIZER!r} (choisir 'char' ou 'bpe')")


def show(text_: str) -> str:
    """Affiche le découpage en tokens, séparés par des barres (script 08)."""
    return "|".join(tokenizer.decode([t]) for t in tokenizer.encode(text_))


sample_line = next((line for line in val_text.splitlines() if len(line.strip()) > 30), val_text[:80])[:80]
print(f"vocab_size : {tokenizer.vocab_size}")
print(show(sample_line))

### Tout le corpus en tenseurs

On encode une fois pour toutes, en `torch.long` : ce sont des **indices** (script 01). L'aller-retour `decode(encode(texte))` doit redonner exactement le texte.

In [ ]:
t0 = time.perf_counter()
train_data = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
val_data = torch.tensor(tokenizer.encode(val_text), dtype=torch.long)
test_data = torch.tensor(tokenizer.encode(test_text), dtype=torch.long)
assert tokenizer.decode(val_data.tolist()) == val_text, "l'aller-retour encode / decode n'est pas exact"
chars_per_token = len(val_text) / len(val_data)  # 1 en mode caractère ; sert à comparer char et BPE
print(f"encodage : {time.perf_counter() - t0:.1f} s")
print(f"train : {len(train_data):,} tokens   val : {len(val_data):,}   test : {len(test_data):,}   "
      f"{chars_per_token:.2f} caractères par token")

Le corpus général s'encode avec `datasets.map` : par lots de documents, en parallèle, et le résultat est gardé en cache tant que le tokenizer ne change pas. Les tokens sont rangés en `int32` (deux fois moins de mémoire que `torch.long` : il y en a plus de cent millions). Sa toute fin (`PRETRAIN_VAL_FRACTION`) sert de validation pendant le pré-entraînement.

In [ ]:
if pretrain is not None:
    t0 = time.perf_counter()
    sep = tokenizer.encode(DOC_SEPARATOR)
    encoded = pretrain.map(lambda batch: {"ids": [ids + sep for ids in tokenizer.encode_batch(batch["text"])]},
                           batched=True, batch_size=2_000, remove_columns=["text"], desc="encodage")
    pretrain_data = torch.from_numpy(np.concatenate(encoded.with_format("numpy")["ids"]).astype(np.int32))
    n = int((1 - PRETRAIN_VAL_FRACTION) * len(pretrain_data))
    pretrain_train_data, pretrain_val_data = pretrain_data[:n], pretrain_data[n:]
    pretrain_chars_per_token = pretrain_chars / len(pretrain_data)
    print(f"corpus général : {len(pretrain_train_data):,} tokens d'entraînement   "
          f"{len(pretrain_val_data):,} de validation   {pretrain_chars_per_token:.2f} caractères par token   "
          f"({time.perf_counter() - t0:.0f} s)")

## 4. Hyperparamètres et lots

On compte en **pas**, pas en époques (script 06) : `max_steps` lots de `batch_size` morceaux de `block_size` tokens, tirés au hasard **avec remise**, et une mesure de validation tous les `eval_every` pas.

Un exemple, c'est un morceau `x` de `block_size` tokens et le même décalé d'un cran `y` : il en commence un à chaque position du texte (le `CharDataset` du script 06).

In [ ]:
def pick_preset(n_tokens: int) -> str:
    """Préréglage conseillé pour un corpus de `n_tokens` tokens d'entraînement."""
    if n_tokens < 250_000:
        return "petit"
    if n_tokens < 2_500_000:
        return "moyen"
    return "gros"


preset = pick_preset(len(pretrain_train_data) if pretrain is not None else len(train_data)) if PRESET == "auto" else PRESET
if preset not in PRESETS:
    raise ValueError(f"PRESET inconnu : {PRESET!r} (choisir parmi {list(PRESETS)} ou 'auto')")
unknown = set(OVERRIDES) - set(TRAINING_DEFAULTS) - set(PRESETS[preset])
if unknown:
    raise ValueError(f"réglages inconnus dans OVERRIDES : {sorted(unknown)}")
hp = {**TRAINING_DEFAULTS, **PRESETS[preset], **OVERRIDES}
block_size, batch_size = hp["block_size"], hp["batch_size"]
# Avec un pré-entraînement : hp pour le corpus général, FINETUNE par-dessus pour l'affinage sur DATA_PATH.
pretrain_hp = {**hp, **PRETRAIN}
finetune_hp = {**hp, **FINETUNE} if pretrain is not None else hp
for name, data in (("validation", val_data), ("test", test_data)):
    if (name == "validation" or TEST_FRACTION > 0) and len(data) <= block_size:
        raise ValueError(f"{name} trop courte ({len(data)} tokens) pour block_size = {block_size} : "
                         "augmenter VAL_FRACTION / TEST_FRACTION, ou baisser block_size dans OVERRIDES")

tokenizer_name = "char" if TOKENIZER == "char" else f"bpe{tokenizer.vocab_size}"
run_name = RUN_NAME or f"{corpus_name}_{tokenizer_name}_{preset}" + ("_pretrained" if pretrain is not None else "")
print(f"préréglage : {preset}{' (auto)' if PRESET == 'auto' else ''}   run : {run_name}")
for key, value in finetune_hp.items():
    print(f"  {key:14} {value}" + (f"   (pré-entraînement : {pretrain_hp[key]})"
                                   if pretrain is not None and pretrain_hp[key] != value else ""))

In [ ]:
class TokenDataset(Dataset):
    """Tous les morceaux de `block_size` tokens d'un texte, avec leurs cibles (le `CharDataset` du script 06).

    Args:
        data: Le texte encodé, un tenseur d'indices.
        block_size: Longueur des morceaux.
    """

    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        chunk = self.data[i : i + self.block_size + 1].long()  # le corpus général est rangé en int32
        return chunk[:-1], chunk[1:]  # x, et y décalé d'un cran


def make_train_loader(dataset: Dataset, n_steps: int, size: int = batch_size) -> DataLoader:
    """Un `DataLoader` qui tire `n_steps` lots de `size` exemples au hasard dans `dataset`."""
    sampler = RandomSampler(dataset, replacement=True, num_samples=n_steps * size)
    return DataLoader(dataset, batch_size=size, sampler=sampler)


def make_val_loader(dataset: Dataset, cfg: dict) -> DataLoader:
    """Des lots de validation ; chaque fois qu'on le parcourt, il tire de nouveaux morceaux."""
    sampler = RandomSampler(dataset, replacement=True, num_samples=cfg["val_batches"] * cfg["batch_size"])
    return DataLoader(dataset, batch_size=cfg["batch_size"], sampler=sampler)


train_dataset = TokenDataset(train_data, block_size)
val_dataset = TokenDataset(val_data, block_size)
val_loader = make_val_loader(val_dataset, finetune_hp)
if pretrain is not None:
    pretrain_dataset = TokenDataset(pretrain_train_data, block_size)
    pretrain_val_loader = make_val_loader(TokenDataset(pretrain_val_data, block_size), pretrain_hp)
x0, y0 = train_dataset[0]
print(f"{len(train_dataset):,} exemples d'entraînement   {len(val_dataset):,} de validation")
print(f"x : {tokenizer.decode(x0[:30].tolist())!r}…")
print(f"y : {tokenizer.decode(y0[:30].tolist())!r}…")

## 5. Le modèle

Le modèle est `ModernGPT`, celui de `v6-591M`. C'est toujours le Transformer du script 06 : un **bloc** = communication (self-attention masquée multi-têtes) puis calcul (MLP), chacun précédé d'une normalisation (*pre-norm*) et entouré d'une connexion résiduelle. Les pièces viennent du speedrun modded-nanoGPT (Keller Jordan et al., 2024-2025), un concours qui cherche à atteindre une loss donnée avec le moins de calcul possible :

| Pièce | GPT du cours (`GPTLanguageModel`) | `ModernGPT` |
|---|---|---|
| Position | un embedding de position appris | **RoPE** : q et k tournent d'un angle proportionnel à leur position, et leur produit ne dépend plus que de la distance entre les deux tokens |
| Normalisation | LayerNorm (moyenne, variance, deux vecteurs appris) | **RMSNorm** sans paramètre : chaque vecteur est ramené à une moyenne quadratique de 1 |
| Attention | softmax(qkᵀ/√d)·v | la même formule, avec q et k normalisés (**QK-norm**, des scores bornés), calculée par l'attention causale de PyTorch (`scaled_dot_product_attention`) |
| MLP | ReLU | **ReLU²** : ReLU puis carré |
| Biais | oui | aucun |
| Raccourcis | les résiduels | en plus, des **connexions en U** : la seconde moitié des blocs reçoit la sortie du bloc symétrique de la première, avec un poids appris |
| Sortie | les logits | des logits **plafonnés** à ±30 : `30 · tanh(logits / 30)` |
| Initialisation | loi normale, écart-type 0,02 | les projections de sortie et la tête partent de **zéro**. Au départ, chaque bloc est l'identité et la loss vaut exactement `ln(vocab_size)` |

Le code est écrit une seule fois, dans `src/super_modele/gpt.py`. Le chat, l'interface et l'entraînement sur TPU s'en servent aussi ; le voici. `forward` renvoie toujours `(logits, loss)`, la convention du script 06, et le masque causal ne se sauvegarde pas. Le GPT du cours reste dans `gpt.py`, comme référence.

In [ ]:
for part in (rms_norm, Rotary, ModernAttention, ModernMLP, ModernBlock, ModernGPT):
    display(Code(inspect.getsource(part), language="python"))

In [ ]:
model_config = {  # les arguments de ModernGPT, et son architecture : le checkpoint en a besoin (script 07)
    "arch": "modern",
    "vocab_size": tokenizer.vocab_size,
    **{key: hp[key] for key in ("block_size", "n_embd", "n_head", "n_layer", "dropout")},
}
torch.manual_seed(SEED)
model = ModernGPT(**{key: value for key, value in model_config.items() if key != "arch"}).to(device)
n_params = sum(p.numel() for p in model.parameters())

model.eval()
xb, yb = next(iter(make_train_loader(train_dataset, 1)))
with torch.no_grad():
    loss0 = model(xb.to(device), yb.to(device))[1].item()
tokens_per_step = batch_size * block_size  # pour l'affinage ; le pré-entraînement a sa propre taille de lot
print(f"{n_params:,} paramètres")
print(f"loss avant entraînement : {loss0:.4f}   (hasard uniforme : ln({tokenizer.vocab_size}) = "
      f"{math.log(tokenizer.vocab_size):.4f} ; la tête part de zéro, toutes les prédictions sont égales)")
if pretrain is not None:
    pre_tokens = pretrain_hp["max_steps"] * pretrain_hp["batch_size"] * block_size
    print(f"pré-entraînement : {pretrain_hp['max_steps']:,} pas × {pretrain_hp['batch_size']} × {block_size} tokens = "
          f"{pre_tokens / 1e6:.0f} M tokens vus, soit {pre_tokens / len(pretrain_train_data):.2f} fois le corpus général")
steps = finetune_hp["max_steps"]
print(f"{'affinage' if pretrain is not None else 'entraînement'} : au plus {steps:,} pas × {batch_size} × {block_size} tokens = "
      f"{steps * tokens_per_step / 1e6:.1f} M tokens vus, soit {steps * tokens_per_step / len(train_data):.1f} fois le train")
if n_params > 20 * (len(pretrain_train_data) if pretrain is not None else len(train_data)):
    print("⚠ modèle très gros pour ce corpus : il risque de le réciter. "
          "Essayer un préréglage plus petit, ou du dropout (OVERRIDES).")

## 6. L'entraînement

La boucle du script 06 : pour chaque pas, un `train_step` ; tous les `eval_every` pas, un `test` sur la validation. L'optimisation est celle de `v6-591M` :

- **deux optimiseurs**. Les matrices des blocs (attention, MLP) apprennent avec **Muon** : le momentum de Nesterov, puis la mise à jour est *orthogonalisée* par quelques itérations de Newton-Schulz. Toutes ses directions ont alors la même amplitude : les directions rares ne sont plus noyées par les dominantes. **NorMuon** y ajoute une échelle par neurone, la moyenne glissante du carré de sa mise à jour, comme Adam mais avec un seul nombre par ligne de la matrice. L'embedding, la tête et les poids des connexions en U, qui ne sont pas des matrices de bloc, restent avec **Adam**. Les learning rates sont ceux du speedrun (0,05 pour NorMuon, 0,6 pour l'embedding, 0,008 pour la tête, 0,04 pour les connexions en U), multipliés par `lr_scale` ;
- **le planning WSD** (*warmup-stable-decay*) : les learning rates montent sur le premier 1 % des pas, restent au plateau, puis descendent linéairement jusqu'à 0 sur les derniers 40 %. Contrairement au cosinus, la durée n'a pas besoin d'être connue d'avance : tant qu'on est sur le plateau, on peut prolonger l'entraînement. C'est ce qui nous a permis d'ajuster la cible de `v6-591M` en cours de route ;
- **pas de dropout ni de gradient clipping**, comme pour `v6-591M`. QK-norm et les logits plafonnés gardent l'entraînement stable. Un modèle pré-entraîné sans dropout ne doit pas en recevoir pour son affinage. Mesuré sur `v6-591M` : avec un dropout de 0,1, la loss d'entraînement du premier pas d'affinage passait à 7,95, contre 2,39 sans ;
- **précision mixte bf16** (`USE_AMP`) : les produits matriciels se font sur 16 bits, plus vite. L'orthogonalisation de Muon se calcule en float32 ;
- **`torch.compile`** (`COMPILE`) : PyTorch fusionne les opérations du modèle en quelques gros calculs sur le GPU. La compilation prend une dizaine de secondes au premier pas ; si elle échoue, l'entraînement continue sans ;
- **arrêt anticipé** : après `patience` mesures sans amélioration de la validation, on s'arrête au lieu d'attendre `max_steps` ;
- **le meilleur modèle est sauvegardé** à chaque mesure où la validation s'améliore. Un modèle qui s'entraîne trop longtemps finit par réciter son corpus (script 06, chapitre 7) : sa loss de validation remonte, mais le checkpoint garde le meilleur moment.

La loss de train affichée est, comme au script 06, la moyenne des pas écoulés depuis la dernière mesure.

**Pré-entraînement puis affinage.** Avec `PRETRAIN_DATA`, l'entraînement se fait en deux étapes, la recette de GPT et de `v6-591M` :

1. **pré-entraînement** sur le corpus général, avec les réglages de `PRETRAIN` (des lots de 128) et le planning WSD. Le modèle apprend la langue : l'orthographe, la grammaire, des faits du monde. Il voit chaque morceau de texte au plus une fois, donc il ne peut pas réciter ; sa validation (la fin du corpus général) baisse tant qu'on le laisse apprendre ;
2. **affinage** sur `DATA_PATH`, à partir du meilleur modèle pré-entraîné, avec des learning rates 10 fois plus petits (`FINETUNE`), une courte montée puis une descente jusqu'à 0 : il garde ce qu'il sait de la langue et prend le format, le style et les personnages des dialogues. La validation et l'arrêt anticipé portent sur les dialogues.

Le bouton stop pendant le pré-entraînement passe directement à l'affinage, avec le meilleur modèle pré-entraîné.

**TensorBoard** : chaque mesure est écrite dans `runs/<run>/`, comme au script 06. Dans un terminal, `uv run tensorboard --logdir runs` (puis http://localhost:6006) suit l'entraînement en direct : les courbes de loss, de bits par caractère, du multiplicateur de learning rate et de vitesse (onglet *Scalars*), et les répliques du modèle à chaque mesure (onglet *Text*).

**En direct dans le notebook** : sous chaque ligne du tableau (`↳`), le modèle en cours d'entraînement répond à la même question, avec la même graine. On le voit passer du charabia aux mots, puis aux phrases.

In [ ]:
for part in (newton_schulz, Muon, modern_optimizers, wsd_schedule):
    display(Code(inspect.getsource(part), language="python"))

In [ ]:
def lr_mult(step: int, cfg: dict) -> float:
    """Multiplicateur des learning rates au pas `step` : le planning WSD (montée, plateau, descente jusqu'à 0)."""
    return wsd_schedule(step / cfg["max_steps"], cfg["warmup"], cfg["cooldown"])


def make_optimizers(model: nn.Module, cfg: dict) -> list:
    """NorMuon pour les matrices des blocs, Adam pour l'embedding, la tête et les connexions en U ; les learning
    rates du speedrun × `lr_scale`."""
    return modern_optimizers(model, **{key: cfg[key] * cfg["lr_scale"] for key in ("muon_lr", "embed_lr", "head_lr", "scalar_lr")},
                             normuon=cfg["normuon"])


def train_step(model: nn.Module, optimizers: list, X: torch.Tensor, Y: torch.Tensor, mult: float) -> float:
    """Un pas d'entraînement sur un lot (script 06).

    Args:
        model: Modèle renvoyant `(logits, loss)`.
        optimizers: NorMuon et Adam, qui se partagent les paramètres du modèle.
        X: Entrées, de forme `(batch_size, block_size)`.
        Y: Cibles, de même forme.
        mult: Le multiplicateur des learning rates (planning WSD).

    Returns:
        La loss du lot.
    """
    model.train()  # dropout actif, s'il y en a
    with autocast():
        _, loss = model(X.to(device), Y.to(device))  # 1. forward, loss comprise
    for optimizer in optimizers:
        optimizer.zero_grad()
    loss.backward()  # 2. backward
    for optimizer in optimizers:
        optimizer.step(mult)  # 3. step : chaque learning rate × le multiplicateur du planning
    return loss.item()


@torch.no_grad()
def test(model: nn.Module, loader: DataLoader) -> float:
    """Mesure la loss moyenne sur les lots du loader, sans apprendre (script 06)."""
    model.eval()  # dropout coupé
    total = 0.0
    for X, Y in loader:
        with autocast():
            _, loss = model(X.to(device), Y.to(device))
        total += loss.item()
    return total / len(loader)


# À chaque mesure, le modèle en cours d'entraînement continue ce début de texte, toujours avec la même graine :
# on voit la même réplique passer du charabia à des phrases.
if CHAT_TEMPLATE is not None:
    PREVIEW_PROMPT = CHAT_TEMPLATE.format(me=CHAT_ME, message="Qu'est-ce que tu fais ?", character=CHAT_CHARACTER)
else:
    PREVIEW_PROMPT = "\n" if "\n" in text else text[0]


@torch.no_grad()
def preview(model: nn.Module, n_tokens: int = 40) -> str:
    """Une courte suite de `PREVIEW_PROMPT` écrite par le modèle, sur une seule ligne."""
    model.eval()
    generator = torch.Generator().manual_seed(SEED)  # graine à part : ne touche pas au tirage des lots
    try:
        idx = torch.tensor([tokenizer.encode(PREVIEW_PROMPT)], device=device)
    except ValueError:  # un caractère du message absent du vocabulaire
        return ""
    for _ in range(n_tokens):
        logits, _ = model(idx[:, -model.block_size :])
        probs = F.softmax(logits[0, -1].float() / 0.8, dim=-1).cpu()
        idx_next = torch.multinomial(probs, num_samples=1, generator=generator).to(device)
        idx = torch.cat((idx, idx_next.view(1, 1)), dim=1)
    out = tokenizer.decode(idx[0].tolist())[len(PREVIEW_PROMPT) :]
    return out.split("\n")[0].strip() if CHAT_TEMPLATE is not None else out.replace("\n", " ⏎ ")


def run_training(model: nn.Module, train_dataset: Dataset, val_loader: DataLoader, cfg: dict,
                 checkpoint_path: Path, title: str, chars_per_tok: float | None = None) -> dict[str, list[float]]:
    """Entraîne pendant `cfg["max_steps"]` pas et garde le meilleur modèle (la boucle du script 06).

    Tous les `eval_every` pas : mesure de la validation, une ligne de suivi, et sauvegarde si la
    validation s'est améliorée. Après `patience` mesures sans amélioration, on arrête : le modèle
    commence à réciter, continuer ne ferait que perdre du temps. Une interruption (bouton stop)
    arrête aussi proprement la boucle.

    Args:
        model: Modèle renvoyant `(logits, loss)`.
        train_dataset: Les exemples d'entraînement.
        val_loader: Les lots de validation.
        cfg: Les réglages : `max_steps`, `learning_rate`, `eval_every`, `patience`…
        checkpoint_path: Fichier du meilleur modèle.
        title: Titre de la courbe.
        chars_per_tok: Caractères par token du corpus, pour les bits par caractère de TensorBoard.

    Returns:
        L'historique des mesures : pas, loss train et val, multiplicateur du learning rate.
    """
    torch.manual_seed(SEED)
    max_steps = cfg["max_steps"]
    for module in model.modules():  # le dropout de l'étape (0 pendant le pré-entraînement)
        if isinstance(module, nn.Dropout):
            module.p = cfg["dropout"]
    optimizers = make_optimizers(model, cfg)
    # torch.compile ne sert qu'à l'entraînement (mêmes poids, seul le calcul change) ; la validation
    # et la génération utilisent le modèle normal.
    step_model = torch.compile(model) if COMPILE else model
    history = {"step": [], "train": [], "val": [], "lr": []}
    best_val = float("inf")
    since_best = 0  # mesures depuis la dernière amélioration
    train_losses = []  # loss des pas depuis la dernière mesure
    # TensorBoard : un dossier par run, repris à zéro (comme au script 06) ; `uv run tensorboard --logdir runs`.
    log_dir = RUNS_DIR / checkpoint_path.stem
    shutil.rmtree(log_dir, ignore_errors=True)
    writer = SummaryWriter(log_dir=str(log_dir))
    writer.add_text("réglages", "  \n".join(f"{key} : {value}" for key, value in cfg.items()))
    tokens_per_step = cfg["batch_size"] * block_size
    start = time.perf_counter()

    def measure(step: int, lr: float) -> None:
        nonlocal best_val, since_best, train_losses
        val_loss = test(model, val_loader)
        train_loss = sum(train_losses) / len(train_losses) if train_losses else float("nan")
        train_losses = []
        history["step"].append(step)
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["lr"].append(lr)
        note = ""
        if val_loss < best_val:
            best_val, since_best = val_loss, 0
            save_checkpoint(checkpoint_path, model, model_config, tokenizer, step=step, val_loss=val_loss,
                            hyperparameters=cfg)
            note = "  ← meilleur, sauvegardé"
        else:
            since_best += 1
        elapsed = time.perf_counter() - start
        remaining = elapsed / step * (max_steps - step) if step else float("nan")
        eta = fmt_duration(remaining) if step else "?"
        print(f"{step:>7,} {train_loss:>8.4f} {val_loss:>8.4f} {lr:>9.3f} {fmt_duration(elapsed):>8} {eta:>8}{note}")
        speaker = f"{CHAT_CHARACTER} : " if CHAT_TEMPLATE is not None else ""
        sample = preview(model)
        print(f"{'':>8}↳ {speaker}{sample}")
        writer.add_scalar("loss/val", val_loss, step)
        if not math.isnan(train_loss):
            writer.add_scalar("loss/train", train_loss, step)
        writer.add_scalar("val/bits par caractère", val_loss / math.log(2) / (chars_per_tok or chars_per_token), step)
        writer.add_scalar("learning rate (multiplicateur)", lr, step)
        if step:
            writer.add_scalar("vitesse/tokens par seconde", step * tokens_per_step / elapsed, step)
        writer.add_text("réplique", speaker + sample, step)
        writer.flush()

    print(f"{'pas':>7} {'train':>8} {'val':>8} {'lr ×':>9} {'écoulé':>8} {'reste':>8}")
    step, lr = 0, lr_mult(0, cfg)
    measure(step, lr)  # le point de départ
    try:
        for step, (X, Y) in enumerate(make_train_loader(train_dataset, max_steps, cfg["batch_size"]), start=1):
            lr = lr_mult(step - 1, cfg)
            try:
                loss = train_step(step_model, optimizers, X, Y, lr)
            except Exception as error:  # torch.compile échoue sur certaines configurations (Mac surtout)
                if step_model is model:
                    raise
                print(f"torch.compile a échoué ({type(error).__name__}) : on continue sans")
                step_model = model
                loss = train_step(step_model, optimizers, X, Y, lr)
            train_losses.append(loss)
            if step % cfg["eval_every"] == 0 or step == max_steps:
                measure(step, lr)
                if since_best >= cfg["patience"]:
                    print(f"arrêt anticipé : la validation ne progresse plus depuis {since_best} mesures")
                    break
    except KeyboardInterrupt:
        print(f"interrompu au pas {step:,}")
        if train_losses:
            measure(step, lr)  # une dernière mesure, pour ne pas perdre les derniers pas
    finally:
        writer.close()
    return history

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / f"{run_name}.safetensors"
pretrain_history = None
if pretrain is not None:
    # Étape 1 : apprendre le français sur le corpus général. Le bouton stop passe directement à l'étape 2.
    pretrain_checkpoint = CHECKPOINT_DIR / f"{PRETRAIN_NAME}_{tokenizer_name}_{preset}.safetensors"
    print(f"=== 1. pré-entraînement sur {PRETRAIN_NAME} ===")
    pretrain_history = run_training(model, pretrain_dataset, pretrain_val_loader, pretrain_hp, pretrain_checkpoint,
                                    f"pré-entraînement : {PRETRAIN_NAME}", pretrain_chars_per_token)
    model, _, pretrained = load_checkpoint(pretrain_checkpoint, device)  # on affine le meilleur modèle pré-entraîné
    print(f"meilleur modèle pré-entraîné : pas {pretrained['step']:,}   val {pretrained['val_loss']:.4f}\n")
    print(f"=== 2. affinage sur {corpus_name} ===")

# L'entraînement sur DATA_PATH : depuis zéro, ou depuis le modèle pré-entraîné (l'affinage).
history = run_training(model, train_dataset, val_loader, finetune_hp, checkpoint_path, run_name)

# On repart du meilleur modèle mesuré, qui n'est pas forcément le dernier.
model, _, best = load_checkpoint(checkpoint_path, device)
print(f"\nmeilleur modèle : pas {best['step']:,}   val {best['val_loss']:.4f}   → {checkpoint_path.relative_to(ROOT)}")

## 7. Courbes et évaluation

Les deux courbes descendent ensemble tant que le modèle apprend la langue. Quand la validation remonte alors que le train continue de baisser, il commence à **réciter** son corpus : c'est le moment que le checkpoint a gardé.

In [ ]:
if pretrain_history is not None:
    fig, ax_pre = plt.subplots(figsize=(13, 3.5))
    ax_pre.plot(pretrain_history["step"], pretrain_history["train"], label="train")
    ax_pre.plot(pretrain_history["step"], pretrain_history["val"], label="validation (corpus général)")
    ax_pre.set_xlabel("pas")
    ax_pre.set_ylabel("loss")
    ax_pre.set_title(f"1. pré-entraînement : {PRETRAIN_NAME}")
    ax_pre.legend()
    ax_pre.grid(alpha=0.3)
    plt.show()

fig, (ax_loss, ax_lr) = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [3, 1]})
ax_loss.plot(history["step"], history["train"], label="train (moyenne des pas)")
ax_loss.plot(history["step"], history["val"], label="validation")
ax_loss.axvline(best["step"], color="gray", ls=":", label=f"meilleur modèle (pas {best['step']:,})")
ax_loss.set_xlabel("pas")
ax_loss.set_ylabel("loss (cross-entropy)")
ax_loss.set_title(("2. affinage : " if pretrain_history is not None else "") + run_name)
ax_loss.legend()
ax_loss.grid(alpha=0.3)
ax_lr.plot(history["step"], history["lr"])
ax_lr.set_xlabel("pas")
ax_lr.set_title("learning rate (multiplicateur, WSD)")
ax_lr.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Pendant l'entraînement, la validation était mesurée sur des morceaux tirés au hasard. Pour les chiffres finaux, on passe sur **toute** la validation et **tout** le test, découpés en morceaux qui se suivent. Le chiffre qui compte est celui du **test** : c'est la première fois que le modèle le voit, et rien n'a été choisi d'après lui. S'il est nettement plus haut que la validation, c'est que nos choix (meilleur checkpoint, réglages) se sont trop ajustés à la validation.

La loss est une moyenne **par token** : elle ne se compare pas entre un modèle caractère et un modèle BPE (un token BPE contient plus de texte, il est plus dur à prédire). On la ramène donc au caractère, en **bits par caractère** : `loss / ln 2 / (caractères par token)`. Deux références : le hasard uniforme (`ln(vocab_size)`) et l'entropie des fréquences de tokens, ce qu'obtient un modèle qui ne regarde pas le contexte (script 06, chapitre 5).

In [ ]:
@torch.no_grad()
def full_loss(model: nn.Module, data: torch.Tensor) -> float:
    """Loss moyenne sur tout un texte encodé, découpé en morceaux de `block_size` tokens qui se suivent."""
    model.eval()
    n_blocks = (len(data) - 1) // block_size
    x = data[: n_blocks * block_size].view(n_blocks, block_size)
    y = data[1 : n_blocks * block_size + 1].view(n_blocks, block_size)
    total = 0.0
    for i in range(0, n_blocks, batch_size):
        with autocast():
            _, loss = model(x[i : i + batch_size].to(device), y[i : i + batch_size].to(device))
        total += loss.item() * len(x[i : i + batch_size])
    return total / n_blocks


def bits_per_char_of(loss: float, encoded: torch.Tensor, raw: str) -> float:
    """La loss par token ramenée au caractère, en bits."""
    return loss / math.log(2) / (len(raw) / len(encoded))


val_loss = full_loss(model, val_data)
val_bpc = bits_per_char_of(val_loss, val_data, val_text)
print(f"validation : loss {val_loss:.4f}   perplexité {math.exp(val_loss):.2f}   {val_bpc:.3f} bits par caractère")
if len(test_data) > block_size:
    test_loss = full_loss(model, test_data)
    test_bpc = bits_per_char_of(test_loss, test_data, test_text)
    print(f"test       : loss {test_loss:.4f}   perplexité {math.exp(test_loss):.2f}   {test_bpc:.3f} bits par caractère"
          "   ← le score du modèle")
else:
    test_loss = test_bpc = None
bits_per_char = test_bpc if test_bpc is not None else val_bpc
freqs = torch.bincount(train_data, minlength=tokenizer.vocab_size).float()
p = freqs[freqs > 0] / freqs.sum()
unigram_entropy = -(p * p.log()).sum().item()
print(f"références : hasard uniforme {math.log(tokenizer.vocab_size):.3f}   "
      f"fréquences seules (unigramme) {unigram_entropy:.3f}")

# Le résultat du run, dans l'onglet HParams de TensorBoard : un tableau de tous les runs, réglages et scores.
hparams = {"corpus": corpus_name, "tokenizer": tokenizer_name, "préréglage": preset,
           "pré-entraînement": PRETRAIN_NAME if pretrain is not None else "aucun", **finetune_hp}
metrics = {"résultat/val bits par caractère": val_bpc, "résultat/val loss": val_loss}
if test_bpc is not None:
    metrics |= {"résultat/test bits par caractère": test_bpc, "résultat/test loss": test_loss}
with SummaryWriter(log_dir=str(RUNS_DIR / checkpoint_path.stem)) as writer:
    writer.add_hparams(hparams, metrics, run_name="résultat")
print(f"résultat → TensorBoard, run {checkpoint_path.stem}, onglet HParams")

## 8. Génération

Le modèle ne fournit que des logits ; **la stratégie de tirage appartient à celui qui génère** (script 07) :

- **glouton** (`temperature=0`) : toujours le token le plus probable. Déterministe, et le texte tombe vite dans une boucle ;
- **température** : on divise les logits avant le softmax. `< 1` concentre la distribution sur les choix sûrs, `> 1` l'aplatit ;
- **top-k** : on ne tire que parmi les `k` tokens les plus probables, la longue traîne des choix absurdes disparaît ;
- **top-p** (*nucleus*) : on garde le plus petit ensemble de tokens dont la probabilité cumulée dépasse `p`. Le nombre de candidats s'adapte à l'incertitude du modèle.

La génération boucle comme au script 06 : prédire, tirer, ajouter, recommencer, en ne gardant que les `block_size` derniers tokens comme contexte.

In [ ]:
def complete(prompt: str, max_new_tokens: int = 500, seed: int | None = SEED, stop: str | None = None,
             **sampling) -> str:
    """Génère la suite de `prompt` avec `generate` (script 07, plus le top-p) et renvoie le texte complet.

    Args:
        prompt: Le début du texte.
        max_new_tokens: Nombre de tokens à générer.
        seed: Graine du tirage ; `None` = un texte différent à chaque appel.
        stop: Si fourni, coupe la suite à la première occurrence de ce texte (un retour à la ligne : une seule ligne).
        **sampling: `temperature`, `top_k`, `top_p`, passés à `generate`.
    """
    if seed is not None:
        torch.manual_seed(seed)
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    out = tokenizer.decode(generate(model, idx, max_new_tokens, **sampling)[0].tolist())
    if stop is not None and (cut := out.find(stop, len(prompt))) != -1:
        out = out[:cut]
    return out


@torch.no_grad()
def next_token_probs(prompt: str, k: int = 10) -> None:
    """Affiche les `k` tokens les plus probables après `prompt` (script 07, chapitre 2)."""
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)[-model.block_size :]], device=device)
    probs = F.softmax(model(idx)[0][0, -1], dim=-1)
    top = probs.topk(min(k, tokenizer.vocab_size))
    print(f"après {prompt[-60:]!r} :")
    for p_, i in zip(top.values, top.indices):
        print(f"  {tokenizer.decode([i.item()])!r:14} {p_.item():.3f}")


display(Code(inspect.getsource(generate), language="python"))
DEFAULT_PROMPT = "\n" if "\n" in text else text[0]  # on démarre d'un retour à la ligne, comme au script 06
next_token_probs(val_text[:60])

In [ ]:
print("--- glouton : toujours le plus probable ---")
print(complete(DEFAULT_PROMPT, 300, temperature=0))
print("\n--- tirage simple (T = 1) ---")
print(complete(DEFAULT_PROMPT, 300, temperature=1.0))
print("\n--- température 0.8, top-p 0.9 ---")
print(complete(DEFAULT_PROMPT, 300, temperature=0.8, top_p=0.9))

### Mesurer l'arbitrage

Comme au script 07, sur environ 1 500 caractères générés par stratégie :

- **mots valides** : la part des mots générés qui existent dans le corpus d'entraînement (la qualité, grossièrement) ;
- **mots distincts** : la part de mots différents parmi les mots générés (la diversité ; le glouton qui boucle est très bas) ;
- **recopié** : la part des passages de 40 caractères du texte généré qu'on retrouve **tels quels** dans le corpus. Proche de 0 : le modèle invente ; élevé : il récite.

In [ ]:
WORD_PATTERN = r"[^\W\d_]+"  # un mot : une suite de lettres, accents compris
corpus_words = set(re.findall(WORD_PATTERN, train_text.lower()))


def word_stats(generated: str) -> tuple[float, float]:
    """Part de mots valides et part de mots distincts dans un texte généré (script 07)."""
    gen_words = re.findall(WORD_PATTERN, generated.lower())
    if not gen_words:
        return 0.0, 0.0
    valid = sum(w in corpus_words for w in gen_words) / len(gen_words)
    distinct = len(set(gen_words)) / len(gen_words)
    return valid, distinct


def copy_rate(generated: str, n: int = 40, n_windows: int = 100) -> float:
    """Part des passages de `n` caractères du texte généré qu'on retrouve tels quels dans le train."""
    starts = range(0, max(1, len(generated) - n), max(1, (len(generated) - n) // n_windows))
    windows = [generated[i : i + n] for i in starts]
    return sum(w in train_text for w in windows) / len(windows)


strategies = {
    "glouton": dict(temperature=0),
    "tirage simple (T=1)": dict(temperature=1.0),
    "température 0.7": dict(temperature=0.7),
    "top-k 10, T=0.8": dict(temperature=0.8, top_k=10),
    "top-p 0.9": dict(temperature=1.0, top_p=0.9),
    "top-p 0.9, T=0.8": dict(temperature=0.8, top_p=0.9),
}
n_eval_tokens = max(100, round(1_500 / chars_per_token))
print(f"{'stratégie':22} {'mots valides':>13} {'mots distincts':>15} {'recopié':>9}")
for name, settings in strategies.items():
    generated = complete(DEFAULT_PROMPT, n_eval_tokens, **settings)[len(DEFAULT_PROMPT) :]
    valid, distinct = word_stats(generated)
    print(f"{name:22} {valid:>13.1%} {distinct:>15.1%} {copy_rate(generated):>9.1%}")

### À vous

Changer le début du texte et les réglages, relancer la cellule. Le modèle ne change pas, seule la façon de tirer dans sa distribution change. Chaque exécution donne un texte différent ; il est aussi enregistré dans `runs/`.

In [ ]:
PROMPT = DEFAULT_PROMPT  # le début du texte, ex. "Marge: Homer, où étais-tu ?\nHomer:"
N_TOKENS = 1_000
TEMPERATURE = 0.8  # 0 = glouton ; < 1 plus sûr, > 1 plus inventif
TOP_K = None  # ex. 10 ; None = pas de coupure
TOP_P = 0.9  # ex. 0.9 ; None = pas de coupure
STOP = None  # ex. "\n" : s'arrêter à la fin de la ligne ; None = générer N_TOKENS tokens

generated = complete(PROMPT, N_TOKENS, seed=None, stop=STOP, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)
print(generated)
(RUNS_DIR / f"{run_name}_generation.txt").write_text(generated, encoding="utf-8")

### Discuter avec le modèle

Si le corpus est un dialogue, `CHAT_TEMPLATE` transforme un message en début de script : `Bart: Papa, je peux avoir un poney ?` puis `Homer:`, et le modèle n'a plus qu'à écrire la réplique suivante ; on s'arrête au retour à la ligne. Comme le corpus contient tous les personnages, on choisit à qui on parle (`character`) et qui parle (`me`). Le modèle écrit plusieurs répliques en parallèle et garde la plus liée au message (`reply`, dans `src/super_modele/gpt.py`) : une réplique passe-partout, probable après n'importe quelle question, perd des points. Il ne « comprend » pas la question pour autant : il continue un script comme ceux qu'il a lus, avec le style et les tics de langage du personnage. Pour discuter dans une vraie interface : `uv run app.py`.

In [ ]:
def chat(message: str, character: str = CHAT_CHARACTER, me: str = CHAT_ME, temperature: float = 0.8,
         candidates: int = 8) -> str:
    """La réplique que `character` répond à `message` dit par `me` : la meilleure de `candidates` essais."""
    prompt = CHAT_TEMPLATE.format(me=me, message=message, character=character)
    return reply(model, tokenizer, prompt, character, device, candidates=candidates, temperature=temperature,
                 echo_penalty=2.0)  # comme l'interface : une réplique qui répète le message perd des points


if CHAT_TEMPLATE is None:
    print("CHAT_TEMPLATE = None : pas de dialogue pour ce corpus")
else:
    CONVERSATIONS = [  # (qui parle, à qui, message) : à changer
        ("Bart", "Homer", "Papa, je peux avoir un poney ?"),
        ("Lenny", "Homer", "Homer, la centrale est en feu !"),
        ("Homer", "Marge", "Marge, qu'est-ce qu'on mange ce soir ?"),
        ("Homer", "Burns", "Monsieur Burns, je voudrais une augmentation."),
        ("Homer", "Moe", "Moe, sers-moi une bière."),
        ("Bart", "Lisa", "Lisa, c'est quoi le sens de la vie ?"),
    ]
    for me, character, message in CONVERSATIONS:
        print(f"{me} → {character} : {message}\n  {character} : {chat(message, character, me)}\n")

## 9. Recharger un modèle sauvegardé

Le checkpoint (`checkpoints/<run>.safetensors`) contient les poids **et**, dans ses métadonnées, tout ce qu'il faut pour s'en servir : la configuration (pour reconstruire l'architecture) et le tokenizer (sans lui, les indices ne veulent rien dire). `load_checkpoint` suffit pour générer sans réentraîner ; c'est ce que font `uv run app.py` (l'interface) et `uv run chat.py` (le terminal).

In [ ]:
print("checkpoints disponibles :", ", ".join(p.name for p in sorted(CHECKPOINT_DIR.glob("*.safetensors"))))
loaded_model, loaded_tokenizer, info = load_checkpoint(checkpoint_path, device)
print(f"rechargé : {info['config']}   val {info['val_loss']:.4f} (pas {info['step']:,})")
torch.manual_seed(SEED)
start_idx = torch.tensor([loaded_tokenizer.encode(DEFAULT_PROMPT)], device=device)
print(loaded_tokenizer.decode(generate(loaded_model, start_idx, 300, temperature=0.8, top_p=0.9)[0].tolist()))

### Le modèle entraîné sur TPU

`v6-591M-ft2` a exactement l'architecture de ce notebook (`ModernGPT`), aux dimensions du préréglage `v6` : 20 blocs de 1 536, 12 têtes de 128, 591 M paramètres. Il a été pré-entraîné sur 7,0 G tokens de français (sous-titres, web, Wikipédia) sur TPU avec `kaggle/train_tpu.py`, puis affiné sur les Simpson avec NorMuon, sans dropout. Score sur le test : 1,086 bits par caractère. S'il est dans `checkpoints/`, on le recharge comme n'importe quel checkpoint, et on lui pose les mêmes questions.

In [ ]:
V6_PATH = CHECKPOINT_DIR / "v6-591M-ft2.safetensors"
if V6_PATH.exists() and CHAT_TEMPLATE is not None:
    v6_model, v6_tokenizer, v6_info = load_checkpoint(V6_PATH, device)
    print(f"{V6_PATH.name} : {type(v6_model).__name__}   {v6_info['config']}")
    print(f"{sum(p.numel() for p in v6_model.parameters()):,} paramètres   meilleur pas de l'affinage : {v6_info['step']}\n")
    for me, character, message in CONVERSATIONS:
        prompt = CHAT_TEMPLATE.format(me=me, message=message, character=character)
        answer = reply(v6_model, v6_tokenizer, prompt, character, device, candidates=8, temperature=0.8, echo_penalty=2.0)
        print(f"{me} → {character} : {message}\n  {character} : {answer}\n")
    del v6_model  # 2,4 Go de poids
else:
    print(f"{V6_PATH.name} absent de checkpoints/ (ou corpus sans dialogue)")

## 10. Comparer les runs

C'est le travail de TensorBoard : `uv run tensorboard --logdir runs`, puis http://localhost:6006.

- *Scalars* : les courbes de tous les runs, à cocher pour les comparer. Les bits par caractère (`val/bits par caractère`) se comparent même entre tokenizers différents, sur le même corpus ;
- *Text* : les répliques du modèle à chaque mesure ;
- *HParams* : un tableau de tous les runs, avec leurs réglages et leurs scores de validation et de test.

## Pour aller plus loin

**Lire les courbes.**

- La validation remonte pendant que le train continue de baisser : le modèle **surapprend** (il récite, la colonne « recopié » monte). Le checkpoint garde déjà le meilleur moment ; pour faire mieux : un préréglage plus petit, moins de pas, plus de données, ou du `dropout` (0,1 à 0,3) pour un modèle entraîné sans pré-entraînement. Un modèle pré-entraîné sans dropout se dérègle si on lui en ajoute à l'affinage.
- Les deux courbes baissent encore à la fin : le modèle **n'a pas fini d'apprendre**. Plus de `max_steps`, ou un préréglage plus gros.
- La loss reste bloquée vers l'entropie unigramme : le modèle n'apprend que les fréquences (script 06, chapitre 5). Baisser `lr_scale`.

**Le tokenizer.** Essayer `TOKENIZER = "bpe"` avec `BPE_VOCAB_SIZE` = 512, 1024, 2048, puis comparer les bits par caractère dans TensorBoard (section 10). Plus le corpus est gros, plus un grand vocabulaire paie.

**Le contexte.** `block_size` est la mémoire du modèle : au-delà, il oublie (script 01). L'attention coûte `T²` : doubler le contexte multiplie par quatre le calcul de l'attention.

**La vitesse.** `COMPILE = True` double à peu près la vitesse sur Mac ; s'il échoue, le notebook le signale et continue sans. Pour les mesures de durée, `COMPILE = False` donne un temps par pas plus régulier.

**Passer à l'échelle.** Le préréglage `v6` est notre modèle final. Il demande des milliards de tokens et des accélérateurs : `kaggle/train_tpu.py` l'entraîne sur TPU (PyTorch/XLA), avec la même architecture, les mêmes optimiseurs et le même planning, sur des lots de 524 288 tokens (1 024 morceaux de 512). Au pré-entraînement, `v6-591M` a vu 7,0 G tokens, soit ~12 tokens par paramètre ; la règle de Chinchilla en conseille une vingtaine.